# E3: giữ hay xóa dòng trùng

**Câu hỏi:** quyết định xóa dòng trùng hoàn toàn trong bước làm sạch ảnh hưởng thế nào đến mô hình và đến con số báo cáo?

**Thiết kế**
- Hai bản dữ liệu: *dedup* (xóa trùng, bản chính, 86.617 dòng) và *keepdup* (giữ dòng trùng, tạo bằng `--keep-duplicates`).
- Cả hai chia theo **đúng ranh giới thời gian** như notebook 02, và cùng chạy hai mô hình tham chiếu mặc định (Logistic Regression, Random Forest), không tinh chỉnh.
- Chấm theo hai cách, vì chúng trả lời hai câu hỏi khác nhau:
  1. **Trên tập kiểm định riêng của từng bản:** con số mà mỗi cách làm sạch sẽ "tự báo cáo". Hai tập kiểm định có số dòng và tỷ lệ hủy khác nhau nên PR-AUC không so sánh thẳng được (mốc của PR-AUC chính là tỷ lệ hủy).
  2. **Trên cùng một tập kiểm định sạch (bản dedup):** so sánh công bằng xem *huấn luyện* trên dữ liệu có dòng trùng làm mô hình tốt lên hay kém đi.
- Chỉ dùng `train` và `val`; không mở `test`.

**Lưu ý:** dòng trùng giống hệt nhau cả ngày đến, nên chúng luôn rơi vào cùng một tập khi chia theo thời gian; bản sao không bao giờ nằm vắt qua train và val. Tác hại của việc giữ trùng vì vậy chỉ là (a) các đơn bị lặp có trọng số lớn hơn khi huấn luyện, (b) các đơn bị lặp được tính nhiều lần khi chấm điểm.

In [1]:
import sys, subprocess; sys.path.insert(0, "..")

import pandas as pd

from src.data import REPO_ROOT, TARGET, load_split
from src.split import time_split
from src.reference import make_reference, REFERENCE
from src.evaluate import evaluate, log_result
from src.compare import paired_bootstrap

AUTHOR = "tuan"
SHOW = ["precision", "recall", "f1", "roc_auc", "pr_auc", "accuracy", "n", "positive_rate"]

RAW = REPO_ROOT / "data" / "raw" / "hotel_bookings.csv"
MAIN = REPO_ROOT / "data" / "interim" / "hotel_bookings_clean.csv"
KEEPDUP = REPO_ROOT / "data" / "interim" / "hotel_bookings_keepdup.csv"
SCRIPT = REPO_ROOT / "src" / "clean_hotel_bookings.py"

## Tạo bản giữ dòng trùng

Chạy lại script làm sạch với công tắc `--keep-duplicates`; mọi bước khác giữ nguyên.

In [2]:
res = subprocess.run([sys.executable, str(SCRIPT), str(RAW), str(KEEPDUP), "--keep-duplicates"],
                     capture_output=True, text=True, check=True)
print(res.stdout)

main = pd.read_csv(MAIN)
dup = pd.read_csv(KEEPDUP)
assert list(main.columns) == list(dup.columns)
print(f"dedup: {len(main):,} dòng | keepdup: {len(dup):,} dòng | khác nhau: {len(dup) - len(main):,} dòng trùng")

Exception in thread Thread-3 (_readerthread):
Traceback (most recent call last):
  File "c:\Users\Maytinh\AppData\Local\Python\pythoncore-3.14-64\Lib\threading.py", line 1082, in _bootstrap_inner
    self._context.run(self.run)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^
  File "c:\Users\Maytinh\AppData\Local\Python\pythoncore-3.14-64\Lib\threading.py", line 1024, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Maytinh\AppData\Local\Python\pythoncore-3.14-64\Lib\subprocess.py", line 1614, in _readerthread
    buffer.append(fh.read())
                  ~~~~~~~^^
  File "c:\Users\Maytinh\AppData\Local\Python\pythoncore-3.14-64\Lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
           ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'charmap' codec can't decode byte 0x90 in position 247: character maps to <undefined>


None
dedup: 86,617 dòng | keepdup: 118,543 dòng | khác nhau: 31,926 dòng trùng


## Chia theo thời gian và so sánh hai bản

In [3]:
parts_main = time_split(main)
parts_dup = time_split(dup)

# kiểm tra: chia bản dedup ở đây phải khớp với file đã chia ở notebook 02
for s in ("train", "val"):
    assert len(parts_main[s]) == len(load_split(s)), f"{s} không khớp notebook 02"

rows = {}
for s in ("train", "val"):
    extra = len(parts_dup[s]) - len(parts_main[s])
    extra_cancel = (parts_dup[s][TARGET].sum() - parts_main[s][TARGET].sum()) / extra
    rows[s] = {"dòng dedup": len(parts_main[s]), "dòng keepdup": len(parts_dup[s]),
               "dòng trùng thêm": extra, "tỷ lệ hủy dedup": parts_main[s][TARGET].mean(),
               "tỷ lệ hủy keepdup": parts_dup[s][TARGET].mean(),
               "tỷ lệ hủy của riêng các dòng trùng": extra_cancel}
pd.DataFrame(rows).T.round(3)

,dòng dedup,dòng keepdup,dòng trùng thêm,tỷ lệ hủy dedup,tỷ lệ hủy keepdup,tỷ lệ hủy của riêng các dòng trùng
train,40688.0,58295.0,17607.0,0.240,0.358,0.63
val,14400.0,19739.0,5339.0,0.284,0.383,0.65


## Huấn luyện và chấm điểm

Với mỗi mô hình tham chiếu: huấn luyện trên *train* của từng bản, rồi chấm trên *val*. Riêng mô hình huấn luyện từ keepdup được chấm hai lần (val riêng và val sạch chung).

In [4]:
def xy(df):
    return df.drop(columns=[TARGET]), df[TARGET]

Xtr_m, ytr_m = xy(parts_main["train"]); Xv_m, yv_m = xy(parts_main["val"])
Xtr_d, ytr_d = xy(parts_dup["train"]);  Xv_d, yv_d = xy(parts_dup["val"])

rows, probas = [], {}
for name in REFERENCE:
    m_dedup = make_reference(name).fit(Xtr_m, ytr_m)
    m_dup = make_reference(name).fit(Xtr_d, ytr_d)

    probas[(name, "dedup")] = m_dedup.predict_proba(Xv_m)[:, 1]
    probas[(name, "keepdup_common")] = m_dup.predict_proba(Xv_m)[:, 1]
    p_own = m_dup.predict_proba(Xv_d)[:, 1]

    for note, y, p in [("dedup", yv_m, probas[(name, "dedup")]),
                       ("keepdup_own_val", yv_d, p_own),
                       ("keepdup_common_val", yv_m, probas[(name, "keepdup_common")])]:
        m = evaluate(y, p)
        log_result(AUTHOR, "E3", name, "val", {**m, "params": "default"}, note=note)
        rows.append({"mô hình": name, "cách làm": note, **m})

e3 = pd.DataFrame(rows).set_index(["mô hình", "cách làm"])[SHOW].round(3)
e3

precision  recall     f1  roc_auc  pr_auc  \
mô hình cách làm                                                        
logreg  dedup                   0.644   0.152  0.246    0.782   0.571   
        keepdup_own_val         0.889   0.388  0.541    0.844   0.803   
        keepdup_common_val      0.612   0.135  0.221    0.778   0.558   
forest  dedup                   0.827   0.112  0.198    0.783   0.590   
        keepdup_own_val         0.956   0.368  0.531    0.851   0.812   
        keepdup_common_val      0.803   0.114  0.200    0.773   0.579   

                            accuracy      n  positive_rate  
mô hình cách làm                                            
logreg  dedup                  0.735  14400          0.284  
        keepdup_own_val        0.747  19739          0.383  
        keepdup_common_val     0.730  14400          0.284  
forest  dedup                  0.741  14400          0.284  
        keepdup_own_val        0.751  19739          0.383  
        keepdup_common_val     0.740  14400          0.284

**Cách đọc bảng**
- `dedup`: huấn luyện và chấm trên bản đã xóa trùng (bản chính của nhóm).
- `keepdup_own_val`: huấn luyện trên bản giữ trùng, chấm trên tập kiểm định *có* dòng trùng. Đây là con số mà quyết định "giữ trùng" sẽ tự báo cáo. Để ý `n` và `positive_rate` khác bản dedup.
- `keepdup_common_val`: huấn luyện trên bản giữ trùng, chấm trên **cùng** tập kiểm định với `dedup`. Chỉ dòng này so sánh công bằng với `dedup`.

## Chênh lệch có lớn hơn nhiễu không?

So sánh `keepdup_common_val` với `dedup` trên cùng các dòng kiểm định bằng bootstrap ghép cặp. Chênh lệch dương nghĩa là huấn luyện trên bản giữ trùng *tốt hơn*.

In [5]:
rows = []
for name in REFERENCE:
    for metric in ("pr_auc", "roc_auc"):
        r = paired_bootstrap(yv_m, probas[(name, "dedup")], probas[(name, "keepdup_common")], metric)
        rows.append({"mô hình": name, "chỉ số": metric, **r})
pd.DataFrame(rows).set_index(["mô hình", "chỉ số"]).round(4)

diff  ci_low  ci_high  ci_excludes_zero
mô hình chỉ số                                            
logreg  pr_auc  -0.0129 -0.0169  -0.0090              True
        roc_auc -0.0043 -0.0064  -0.0021              True
forest  pr_auc  -0.0105 -0.0132  -0.0077              True
        roc_auc -0.0098 -0.0116  -0.0081              True

Bootstrap này chỉ đo nhiễu do chọn mẫu kiểm định, không đo nhiễu do huấn luyện (Random Forest đổi seed sẽ lệch thêm một chút). Nếu khoảng tin cậy chứa 0 thì kết luận đúng là "chưa thấy khác biệt rõ", không phải "hai cách bằng nhau".

## Kết luận E3 

- Con số báo cáo thay đổi rất nhiều, nhưng không phải do mô hình tốt lên. Nếu giữ trùng và chấm trên tập kiểm định riêng thì PR-AUC là 0,803 (logreg) và 0,812 (forest). Bản dedup chỉ được 0,571 và 0,590. ROC-AUC tăng từ khoảng 0,78 lên 0,84–0,85, precision của logreg tăng từ 0,64 lên 0,89.
- Nguyên nhân chính là các dòng trùng có tỷ lệ hủy rất cao, khoảng 63% (train) và 65% (val), trong khi bản dedup chỉ có 24% và 28,4%. Việc giữ trùng làm tỷ lệ hủy của val tăng từ 28,4% lên 38,3%, và PR-AUC có mốc chính là tỷ lệ hủy.
- ROC-AUC không phụ thuộc tỷ lệ hủy mà vẫn tăng khoảng 0,06–0,07. Vì vậy phần còn lại nhiều khả năng đến từ việc cùng một đơn bị tính nhiều lần khi chấm điểm. Cơ chế này chưa được kiểm chứng riêng.
- Trên cùng một tập kiểm định sạch, huấn luyện có dòng trùng làm mô hình hơi kém đi: PR-AUC giảm 0,013 với logreg (khoảng tin cậy 95% là [−0,017; −0,009]) và 0,011 với forest ([−0,013; −0,008]). ROC-AUC giảm 0,004 và 0,010. Khoảng tin cậy đều không chứa 0, nhưng mức chênh chỉ khoảng 1 điểm.
- Khoảng tin cậy này chỉ đo nhiễu do chọn mẫu kiểm định, không đo nhiễu do huấn luyện (Random Forest đổi seed sẽ lệch thêm).
- Tác động lên E8. Nếu giữ trùng, tỷ lệ hủy toàn bộ dữ liệu là 37,3% thay vì 27,7%. Chi phí của phương án không overbook chính là số đơn hủy, nên giữ trùng làm overbooking trông có lợi hơn thực tế. Xóa trùng giúp kết luận E8 thận trọng hơn.
- Lựa chọn của nhóm là xóa trùng, và bằng chứng ủng hộ. Số liệu trung thực hơn và mô hình không tệ đi trên tập sạch.
- Giới hạn: dữ liệu không có mã đặt phòng, nên không biết dòng trùng là bản ghi lặp hay hai đơn thật giống nhau (ví dụ đặt đoàn). Nếu có đơn thật bị xóa, tỷ lệ hủy của dữ liệu sạch có thể thấp hơn thực tế. Hai mô hình tham chiếu chạy tham số mặc định, trên một lần chia theo thời gian.